In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from autogluon.tabular import TabularDataset, TabularPredictor

In [28]:
df = pd.read_csv("../../data/creditcard.csv")

In [29]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)

# =========================
# DIVISÃO TREINO/TESTE
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## SMOTEENN

In [30]:
from imblearn.combine import SMOTEENN
from collections import Counter

def apply_smoteenn(X_train, y_train):
    sme = SMOTEENN(random_state=42)
    X_train, y_train = sme.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

## ADASYN

In [31]:
from imblearn.over_sampling import ADASYN

def apply_adasyn(X_train, y_train):
    adasyn = ADASYN(random_state=42)
    X_train, y_train = adasyn.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

## BorderlineSMOTE

In [32]:
from imblearn.over_sampling import BorderlineSMOTE

def apply_borderline_smote(X_train, y_train):
    borderline_smote = BorderlineSMOTE(random_state=42)
    X_train, y_train = borderline_smote.fit_resample(X_train, y_train)
    print('Resampled dataset shape %s' % Counter(y_train))
    
    return X_train, y_train

## KmeansSMOTE

In [33]:
from imblearn.over_sampling import KMeansSMOTE
from sklearn.preprocessing import StandardScaler

def apply_kmeans_smote(X_train, y_train):
    # Clustering and nearest-neighbor distances need comparable feature scales.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    # The default 0.1 cluster balance threshold is too strict for this rare class.
    sampler = KMeansSMOTE(
        random_state=42,
        cluster_balance_threshold=0.0,
        k_neighbors=5,
    )
    X_resampled_scaled, y_resampled = sampler.fit_resample(X_scaled, y_train)

    # Return synthetic features on the original scale for AutoGluon.
    X_resampled = scaler.inverse_transform(X_resampled_scaled)
    X_resampled = pd.DataFrame(X_resampled, columns=X_train.columns)
    y_resampled = pd.Series(y_resampled, name=getattr(y_train, "name", None))

    print('Resampled dataset shape %s' % Counter(y_resampled))
    return X_resampled, y_resampled


In [34]:
X_train_original, y_train_original = X_train.copy(), y_train.copy()

In [35]:
    # =========================
    # INSTALAR AUTOGLUON
    # =========================
    #%pip install autogluon

for resampling_method in ['smoteenn', 'adasyn', 'borderline_smote', 'kmeans_smote']:
    print(f"\n===== MÉTODO DE REAMOSTRAGEM: {resampling_method.upper()} =====")
    if resampling_method == 'smoteenn':
        X_train, y_train = apply_smoteenn(X_train_original, y_train_original)
    elif resampling_method == 'adasyn':
        X_train, y_train = apply_adasyn(X_train_original, y_train_original)
    elif resampling_method == 'borderline_smote':
        X_train, y_train = apply_borderline_smote(X_train_original, y_train_original)
    elif resampling_method == 'kmeans_smote':
        try:
            X_train, y_train = apply_kmeans_smote(X_train_original, y_train_original)
        except RuntimeError as e:
            if "No clusters found with sufficient samples" not in str(e):
                raise  # Let other RuntimeErrors surface
            print(f"Skipping {resampling_method}: {e}")
            continue
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    train_data = TabularDataset(train_df)
    test_data = TabularDataset(test_df)

    # =========================
    # AUTOGLUON
    # =========================
    predictor = TabularPredictor(
        label='Class',
        eval_metric='f1'
    ).fit(
        train_data=train_data,
        presets='medium_quality',
        time_limit=300  # Orçamento de tempo em segundos
    )

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    leaderboard = predictor.leaderboard(test_data, display=True)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    resultados = []

    X_test_ag = test_df.drop(columns=['Class'])
    y_true = test_df['Class'].values

    # Iterar sobre todos os modelos gerados (incluindo Ensembles)
    for model_name in predictor.model_names():

        # Previsão específica para cada modelo
        preds = predictor.predict(X_test_ag, model=model_name)
        y_pred = preds.astype(int).values

        # Métricas
        acc = round(accuracy_score(y_true, y_pred), 4)
        precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
        f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
        f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

        resultados.append({
            "Modelo": model_name,
            "Tipo": model_name.split('_')[0],
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "F1-score Weighted": f1_weighted
        })

    # =========================
    # RESULTADO FINAL
    # =========================
    df_resultados = pd.DataFrame(resultados)

    print("\n===== MÉTRICAS DE TODOS OS MODELOS =====")
    print(df_resultados.sort_values(by="F1-score Weighted", ascending=False))


===== MÉTODO DE REAMOSTRAGEM: SMOTEENN =====


No path specified. Models will be saved in: "AutogluonModels/ag-20261015_213755"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.81 GB / 16.00 GB (17.6%)
Disk Space Avail:   211.72 GB / 460.38 GB (46.0%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


Resampled dataset shape Counter({1: 218370, 0: 209282})


Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/TCC/notebooks/balanced/AutogluonModels/ag-20261015_213755"
Train Data Rows:    427652
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    2778.51 MB
	Train Data (Original)  Memory Usage: 97.88 MB (3.5% of available memory)
	Inferring data type of each feature based on column values. Se


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch    0.783019   1.000000          f1        0.283374       0.025974  55.665305                 0.283374                0.025974          55.665305            1       True          6
1             CatBoost    0.704918   0.999542          f1        0.015082       0.003419  16.646502                 0.015082                0.003419          16.646502            1       True          3
2              XGBoost    0.661479   0.999771          f1        0.056342       0.006630   3.579693                 0.056342                0.006630           3.579693            1       True          5
3      NeuralNetFastAI    0.617647   1.000000          f1        0.168797       0.020889  87.419324                 0.168797                0.020889          87.41

No path specified. Models will be saved in: "AutogluonModels/ag-20261015_214101"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.40 GB / 16.00 GB (21.2%)
Disk Space Avail:   211.62 GB / 460.38 GB (46.0%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/TCC/notebooks/balanced/AutogluonModels/ag-20261015_214101"
Train Data Rows:    454797
Train Data Columns: 30
Label Colum

... (7 outputs left out)

No path specified. Models will be saved in: "AutogluonModels/ag-20261015_215134"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.13
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       2.72 GB / 16.00 GB (17.0%)
Disk Space Avail:   208.26 GB / 460.38 GB (45.2%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


Resampled dataset shape Counter({1: 227458, 0: 227451})


Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "/Users/lannaces/Documents/TCC/notebooks/balanced/AutogluonModels/ag-20261015_215134"
Train Data Rows:    454909
Train Data Columns: 30
Label Column:       Class
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.int64(0), np.int64(1)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    2903.00 MB
	Train Data (Original)  Memory Usage: 104.12 MB (3.6% of available memory)
	Inferring data type of each feature based on column values. S


===== LEADERBOARD =====
                 model  score_test  score_val eval_metric  pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0             LightGBM    0.759259    0.99956          f1        0.285252       0.008984    7.705735                 0.285252                0.008984           7.705735            1       True          2
1             CatBoost    0.710744    0.99956          f1        0.255333       0.035557  279.510521                 0.255333                0.035557         279.510521            1       True          3
2           LightGBMXT    0.682540    0.99978          f1        0.960259       0.010605    9.992067                 0.960259                0.010605           9.992067            1       True          1
3  WeightedEnsemble_L2    0.682540    0.99978          f1        1.073063       0.013760   11.667574                 0.112804                0.003155          